In [10]:
import os
import ast
import gurobipy as gp
from gurobipy import GRB

os.environ['GRB_LICENSE_FILE'] = r'C:\Users\anabe\gurobi.lic'


In [11]:
# Caminho da pasta com os dados
pasta_dados = r'C:\Users\anabe\OneDrive\Área de Trabalho\IC\Dados'

def ler_arquivo(nome_arquivo, tipo='lista'):
    
    caminho = os.path.join(pasta_dados, nome_arquivo)
    dados = {}
    
    # Verifica se o arquivo existe
    if not os.path.exists(caminho):
        print(f"ATENÇÃO: Arquivo não encontrado -> {caminho}")
        return dados
        
    with open(caminho, 'r', encoding='utf-8') as f:
        for linha in f:
            linha = linha.strip()
            if not linha: continue
            
            # Separa o ID da instância do resto do conteúdo (divide apenas na primeira vírgula)
            partes = linha.split(',', 1)
            id_inst = int(partes[0])
            resto = partes[1].strip()
            
            if tipo == 'lista':
                # Converte as strings "[...]" ou "[(...)]" em listas/tuplas reais do Python
                dados[id_inst] = ast.literal_eval(resto)
            elif tipo == 'parametro':
                # O número de alunos é o primeiro valor logo após o ID da instância
                num_alunos = int(resto.split(',')[0])
                dados[id_inst] = num_alunos
                
    return dados

# Carregando todos os dados em dicionários ANTES de iniciar o loop
dict_conflitos = ler_arquivo('grafo.txt', tipo='lista')
dict_frente = ler_arquivo('grafo_frente.txt', tipo='lista')
dict_atras = ler_arquivo('grafo_tras.txt', tipo='lista')
#dict_J = ler_arquivo('grafo_proximidade.txt', tipo='lista')
dict_fileiras = ler_arquivo('grafo_vet_com_vazios.txt', tipo='lista')
dict_parametros = ler_arquivo('grafo_info.txt', tipo='parametro')


# Loop para resolver as 30 instâncias (1 a 30)
for id_instancia in range(1, 31):
    print(f"{'='*50}")
    print(f" RESOLVENDO {id_instancia} ")
    print(f"{'='*50}")

    # Verifica se a instância existe em todos os dicionários lidos
    try:
        # 1. DADOS
        conflitos = dict_conflitos[id_instancia]
        frente = dict_frente[id_instancia]
        atras = dict_atras[id_instancia]
        #J = dict_J[id_instancia]
        fileiras_cap = dict_fileiras[id_instancia]
        num_alunos = dict_parametros[id_instancia]
    except KeyError:
        print(f"Dados incompletos para a Instância {id_instancia}. Pulando para a próxima...\n")
        continue

    alunos = list(range(num_alunos))
    num_fileiras = len(fileiras_cap)
    
    #parametros
    P = 2*((len(conflitos) * max(fileiras_cap)) + 1)
    d_min = 2                

    # 2. VARIÁVEIS
    model = gp.Model(f"Instancia_{id_instancia}")
    
    # Opcional: silenciar os logs massivos do Gurobi (descomente a linha abaixo se quiser)
    # model.setParam('OutputFlag', 0)

    # x[i, L, k] é 1 se aluno i está na fileira L posição k
    x = {}
    for i in alunos:
        for L in range(num_fileiras):
            for k in range(fileiras_cap[L]):
                x[i, L, k] = model.addVar(vtype=GRB.BINARY, name=f"x_{i}_{L}_{k}")

    # y[i, j] é 1 se a aresta de conflito entre i e j está ativa
    y = {}
    for (i, j) in conflitos:
        y[i, j] = model.addVar(vtype=GRB.BINARY, name=f"y_{i}_{j}")

    # w[L, i, j, k, z] lineariza o produto x[i,L,k] * x[j,L+1,z]
    w = {}
    for L in range(num_fileiras - 1):
        for (i, j) in conflitos:
            for k in range(fileiras_cap[L]):
                for z in range(fileiras_cap[L+1]):
                    w[L, i, j, k, z] = model.addVar(vtype=GRB.BINARY, name=f"w_{L}_{i}_{j}_{k}_{z}")

    # 3. F.O.
    # max dist horizontal e min numero de conflitos
    obj = gp.quicksum(
        (abs(z - k) * w[L, i, j, k, z]) - (P * y[i, j])
        for L in range(num_fileiras - 1)
        for (i, j) in conflitos
        for k in range(fileiras_cap[L])
        for z in range(fileiras_cap[L+1])
    )
    model.setObjective(obj, GRB.MAXIMIZE)

    # 4. S.T.
    # cada aluno deve ocupar um assento
    for i in alunos:
        model.addConstr(gp.quicksum(x[i, L, k] for L in range(num_fileiras) for k in range(fileiras_cap[L])) == 1)

    # cada assento deve ter no máximo um aluno
    for L in range(num_fileiras):
        for k in range(fileiras_cap[L]):
            model.addConstr(gp.quicksum(x[i, L, k] for i in alunos) <= 1)

    # se i e j em conflito estão na mesma fileira a dist >= d_min
    for (i, j) in conflitos:
        for L in range(num_fileiras):
            for k in range(fileiras_cap[L]):
                for z in range(k + 1, fileiras_cap[L]):
                    model.addConstr(z - k >= (x[i, L, k] + x[j, L, z] - 1) * d_min)
                    model.addConstr(z - k >= (x[j, L, k] + x[i, L, z] - 1) * d_min)

    # definição de aresta ativa (y)
    for (i, j) in conflitos:
        for L in range(num_fileiras - 1):
            for k in range(fileiras_cap[L]):
                for z in range(fileiras_cap[L+1]):
                    # i em L,k e j em L+1,z -> aresta fica ativa
                    model.addConstr(y[i, j] >= x[i, L, k] + x[j, L+1, z] - 1)
                    model.addConstr(y[i, j] >= x[j, L, k] + x[i, L+1, z] - 1)

                    # def W
                    model.addConstr(w[L, i, j, k, z] <= x[i, L, k])
                    model.addConstr(w[L, i, j, k, z] <= x[j, L+1, z])
                    model.addConstr(w[L, i, j, k, z] >= x[i, L, k] + x[j, L+1, z] - 1)

    # alunos na lista frente: k=1 ou k=2
    for i_frente in frente:
        model.addConstr(gp.quicksum(x[i_frente, L, k] for L in range(num_fileiras) for k in [0, 1]) == 1)

    # alunos na lista atras: duas últimas posições
    for i_atras in atras:
        model.addConstr(gp.quicksum(x[i_atras, L, k] for L in range(num_fileiras) for k in [fileiras_cap[L]-2, fileiras_cap[L]-1]) == 1)
        
    # proximidade obrigatória (J)
#    for (i, j) in J:
#        for L in range(num_fileiras):
#            for k in range(fileiras_cap[L]):
#                vizinhos = []
#               
#                # Lado
#                if k > 0: 
#                    vizinhos.append(x[j, L, k-1]) # Esquerda
#                if k < fileiras_cap[L] - 1: 
#                    vizinhos.append(x[j, L, k+1]) # Direita
#                    
#                # Frente e Trás
#                if L > 0 and k < fileiras_cap[L-1]: 
#                    vizinhos.append(x[j, L-1, k]) # Frente
#                if L < num_fileiras - 1 and k < fileiras_cap[L+1]: 
#                    vizinhos.append(x[j, L+1, k]) # Atrás
#                
#                # Se o aluno i sentar em (L, k), o aluno j DEVE estar em um dos vizinhos
#                model.addConstr(x[i, L, k] <= gp.quicksum(vizinhos), name=f"dupla_{i}_{j}_{L}_{k}")

    # 5. RESULTADO
    model.Params.TimeLimit = 300
    model.optimize()

   # Tradução do Status do Gurobi
    status_map = {
        GRB.OPTIMAL: "SOLUÇÃO ÓTIMA ENCONTRADA",
        GRB.TIME_LIMIT: "LIMITE DE TEMPO ATINGIDO",
        GRB.INFEASIBLE: "MODELO INVIÁVEL",
        GRB.INF_OR_UNBD: "INVIÁVEL OU ILIMITADO"
    }
    status_msg = status_map.get(model.status, f"STATUS: {model.status}")

    print(f"\nINSTÂNCIA {id_instancia:02d} | Status: {status_msg}")
    print(f"Tempo de Execução: {model.Runtime:.2f} s")
    
    
    # 2. Identificação de Alunos em Fileiras Adjacentes
    if model.SolCount > 0:
        
        print(f"Valor da FO: {model.ObjVal:.2f}")
        # Contamos quantas variáveis y[i, j] o Gurobi ativou (valor > 0.5)
        arestas_ativas = sum(1 for (i, j) in conflitos if y[i, j].X > 0.5)
        
        if arestas_ativas > 0:
            print(f"RESULTADO: {arestas_ativas} aresta(s) de conflito permanecem ativa(s).")
        else:
            print("RESULTADO: Zero conflitos entre fileiras adjacentes.")
    
    print("-" * 50)
    
    #if model.status == GRB.OPTIMAL or model.SolCount > 0:
       # print(f"\n--- MAPA DA SALA (Instância {id_instancia}) ---")
       # for L in range(num_fileiras):
          #  fila = []
           # for k in range(fileiras_cap[L]):
             #   aluno_no_assento = "  "
              #  for i in alunos:
                #    if x[i, L, k].X > 0.5:
                 #       aluno_no_assento = f"{i:2}"
                  #      break
                #fila.append(f"[{aluno_no_assento}]")
            #print(f"Fileira {L}: {' '.join(fila)}")
        #print("\n")
    #else:
     #   print(f"\nNenhuma solução viável encontrada para a Instância {id_instancia}.\n")

 RESOLVENDO 1 
Set parameter TimeLimit to value 300
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-1035G1 CPU @ 1.00GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 23138 rows, 5037 columns and 60206 nonzeros
Model fingerprint: 0xa60f8ea4
Variable types: 0 continuous, 5037 integer (5037 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [1e+00, 5e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+00]
Found heuristic solution: objective -234719.0000
Presolve removed 14201 rows and 2206 columns
Presolve time: 0.31s
Presolved: 8937 rows, 2831 columns, 25051 nonzeros
Variable types: 0 continuous, 2831 integer (2831 binary)

Root relaxation: objective 1.537455e+02, 5859 iterations, 0.33 seconds (0.36 work units)

    Nodes    |    Current N